In [2]:
import pandas as pd

In [3]:
map_ds_path = "../data/raw/usmetros.csv"
df = pd.read_csv(map_ds_path)
df.shape

(387, 11)

In [4]:
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows

In [5]:
df.head(2)

,metro_fips,metro,metro_ascii,metro_full,county_name,county_fips,state_id,state_name,lat,lng,population
0,35620,New York,New York,"New York-Newark-Jersey City, NY-NJ",Suffolk,36103,NY,New York,40.7222,-74.0225,19940274
1,31080,Los Angeles,Los Angeles,"Los Angeles-Long Beach-Anaheim, CA",Los Angeles,6037,CA,California,34.2215,-118.1494,12927614


In [6]:
df['metro_full'].value_counts()

metro_full
New York-Newark-Jersey City, NY-NJ                1
Alexandria, LA                                    1
Janesville-Beloit, WI                             1
Terre Haute, IN                                   1
Kenosha, WI                                       1
Pueblo, CO                                        1
Odessa, TX                                        1
Waterloo-Cedar Falls, IA                          1
Homosassa Springs, FL                             1
La Crosse-Onalaska, WI-MN                         1
Idaho Falls, ID                                   1
Bloomington, IL                                   1
Sebastian-Vero Beach-West Vero Corridor, FL       1
Oshkosh-Neenah, WI                                1
Eau Claire, WI                                    1
Muskegon-Norton Shores, MI                        1
Greenville, NC                                    1
Redding, CA                                       1
El Centro, CA                                     1
A

### We will use the longitude and latitude
- Instead of city_full we will use the lon-lat vales, because there are 30 citys and encodeing will be difficut
    - Numerical encoding will be biased on the importance of the values
    - One hot encoding will be very sparse feature set
- Make sure correspondance of metro_full in simplemaps dataset and the HouseTS dataset are same

In [7]:
train_data = "../data/raw/train.csv"
train_df = pd.read_csv(train_data)
train_df.head(2)

,date,median_sale_price,median_list_price,median_ppsf,median_list_ppsf,homes_sold,pending_sales,new_listings,inventory,median_dom,avg_sale_to_list,sold_above_list,off_market_in_two_weeks,city,zipcode,year,bank,bus,hospital,mall,park,restaurant,school,station,supermarket,Total Population,Median Age,Per Capita Income,Total Families Below Poverty,Total Housing Units,Median Rent,Median Home Value,Total Labor Force,Unemployed Population,Total School Age Population,Total School Enrollment,Median Commute Time,price,city_full
0,2012-03-31,46550.0,217450.0,31.813674,110.183666,14.0,23.0,44.0,64.0,59.5,0.943662,0.142857,0.043478,ATL,30002,2012,12.0,2.0,4.0,1.0,60.0,45.0,57.0,4.0,7.0,5811.0,36.3,33052.0,5811.0,2677.0,710.0,279500.0,3171.0,460.0,5408.0,5408.0,2492.0,200773.999557,Atlanta-Sandy Springs-Alpharetta
1,2012-03-31,200000.0,7500.0,104.931794,79.265873,1.0,1.0,1.0,2.0,290.0,0.909091,0.000000,0.000000,PGH,15469,2012,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,2441.0,41.8,20241.0,2385.0,1108.0,641.0,94600.0,1171.0,52.0,2376.0,2376.0,1018.0,105863.681174,Pittsburgh


In [9]:
train_df['city_full'].value_counts()

city_full
New York-Newark-Jersey City            78020
Chicago-Naperville-Elgin               35344
Los Angeles-Long Beach-Anaheim         33840
Philadelphia-Camden-Wilmington         31396
DC_Metro                               29516
Pittsburgh                             27824
Boston-Cambridge-Newton                25568
Dallas-Fort Worth-Arlington            23594
Houston-The Woodlands-Sugar Land       20586
Minneapolis-St. Paul-Bloomington       20398
Detroit-Warren-Dearborn                20022
St. Louis                              19834
Atlanta-Sandy Springs-Alpharetta       19082
Miami-Fort Lauderdale-Pompano Beach    17014
San Francisco-Oakland-Berkeley         15604
Seattle-Tacoma-Bellevue                14664
Phoenix-Mesa-Chandler                  14006
Cincinnati                             14006
Baltimore-Columbia-Towson              13818
Riverside-San Bernardino-Ontario       13724
Tampa-St. Petersburg-Clearwater        12126
Denver-Aurora-Lakewood                 11750


### Find mismatch
- we need the city_full name correspond to metro_full so the we can get the exact Long-Lat
- For this I'll use a case insensitive comparison

In [12]:
# Create normalized versions for comparison
train_df['city_norm'] = train_df['city_full'].str.lower().str.strip()
df['metro_norm'] = df['metro_full'].str.lower().str.strip()

# Find mismatches
missing_in_df_norm = train_df[~train_df['city_norm'].isin(df['metro_norm'])]
missing_in_train_df_norm = df[~df['metro_norm'].isin(train_df['city_norm'])]

print("Original names in train_df not found in df:")
print(missing_in_train_df_norm[['metro_full']].to_string(index=False))

Original names in train_df not found in df:
                                    metro_full
            New York-Newark-Jersey City, NY-NJ
            Los Angeles-Long Beach-Anaheim, CA
               Chicago-Naperville-Elgin, IL-IN
               Dallas-Fort Worth-Arlington, TX
            Houston-Pasadena-The Woodlands, TX
     Miami-Fort Lauderdale-West Palm Beach, FL
  Washington-Arlington-Alexandria, DC-VA-MD-WV
             Atlanta-Sandy Springs-Roswell, GA
   Philadelphia-Camden-Wilmington, PA-NJ-DE-MD
                     Phoenix-Mesa-Chandler, AZ
                Boston-Cambridge-Newton, MA-NH
          Riverside-San Bernardino-Ontario, CA
             San Francisco-Oakland-Fremont, CA
                   Detroit-Warren-Dearborn, MI
                   Seattle-Tacoma-Bellevue, WA
       Minneapolis-St. Paul-Bloomington, MN-WI
           Tampa-St. Petersburg-Clearwater, FL
            San Diego-Chula Vista-Carlsbad, CA
                  Denver-Aurora-Centennial, CO
                

In [38]:
DF1 = train_df['city_full'].unique()
DF2 = df['metro_full'].unique()

In [39]:
DF1

array(['Atlanta-Sandy Springs-Alpharetta', 'Pittsburgh',
       'Boston-Cambridge-Newton', 'Tampa-St. Petersburg-Clearwater',
       'Baltimore-Columbia-Towson', 'Portland-Vancouver-Hillsboro',
       'Philadelphia-Camden-Wilmington', 'New York-Newark-Jersey City',
       'Chicago-Naperville-Elgin', 'Orlando-Kissimmee-Sanford',
       'Seattle-Tacoma-Bellevue', 'San Francisco-Oakland-Berkeley',
       'San Diego-Chula Vista-Carlsbad', 'Austin-Round Rock-Georgetown',
       'St. Louis', 'Sacramento-Roseville-Folsom',
       'Phoenix-Mesa-Chandler', 'Riverside-San Bernardino-Ontario',
       'San Antonio-New Braunfels', 'Detroit-Warren-Dearborn',
       'Cincinnati', 'Houston-The Woodlands-Sugar Land',
       'Charlotte-Concord-Gastonia', 'Denver-Aurora-Lakewood',
       'Los Angeles-Long Beach-Anaheim', 'DC_Metro',
       'Dallas-Fort Worth-Arlington', 'Minneapolis-St. Paul-Bloomington',
       'Las Vegas-Henderson-Paradise',
       'Miami-Fort Lauderdale-Pompano Beach'], dtype=object)

In [33]:
DF2

array(['New York-Newark-Jersey City, NY-NJ',
       'Los Angeles-Long Beach-Anaheim, CA',
       'Chicago-Naperville-Elgin, IL-IN',
       'Dallas-Fort Worth-Arlington, TX',
       'Houston-Pasadena-The Woodlands, TX',
       'Miami-Fort Lauderdale-West Palm Beach, FL',
       'Washington-Arlington-Alexandria, DC-VA-MD-WV',
       'Atlanta-Sandy Springs-Roswell, GA',
       'Philadelphia-Camden-Wilmington, PA-NJ-DE-MD',
       'Phoenix-Mesa-Chandler, AZ', 'Boston-Cambridge-Newton, MA-NH',
       'Riverside-San Bernardino-Ontario, CA',
       'San Francisco-Oakland-Fremont, CA', 'Detroit-Warren-Dearborn, MI',
       'Seattle-Tacoma-Bellevue, WA',
       'Minneapolis-St. Paul-Bloomington, MN-WI',
       'Tampa-St. Petersburg-Clearwater, FL',
       'San Diego-Chula Vista-Carlsbad, CA',
       'Denver-Aurora-Centennial, CO', 'Orlando-Kissimmee-Sanford, FL',
       'Charlotte-Concord-Gastonia, NC-SC',
       'Baltimore-Columbia-Towson, MD', 'St. Louis, MO-IL',
       'San Antonio-New Braun

In [40]:
# Create DF1
DF1 = pd.DataFrame({'city_full': train_df['city_full'].unique()})
DF2 = pd.DataFrame({'metro_full': df['metro_full'].unique()})

In [36]:
DF2

,metro_full
0,"New York-Newark-Jersey City, NY-NJ"
1,"Los Angeles-Long Beach-Anaheim, CA"
2,"Chicago-Naperville-Elgin, IL-IN"
3,"Dallas-Fort Worth-Arlington, TX"
4,"Houston-Pasadena-The Woodlands, TX"
5,"Miami-Fort Lauderdale-West Palm Beach, FL"
6,"Washington-Arlington-Alexandria, DC-VA-MD-WV"
7,"Atlanta-Sandy Springs-Roswell, GA"
8,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD"
9,"Phoenix-Mesa-Chandler, AZ"


In [42]:
# Extract first word, convert to lowercase
DF1['first_word'] = DF1['city_full'].str.split('-').str[0].str.lower()
DF2['first_word'] = DF2['metro_full'].str.split('-').str[0].str.lower()

# Handle special cases (like "St." and "DC")
DF1['first_word'] = DF1['first_word'].str.replace('st.', 'saint', regex=False)
DF2['first_word'] = DF2['first_word'].str.replace('st.', 'saint', regex=False)

# Special handling for DC_Metro
DF1.loc[DF1['city_full'] == 'DC_Metro', 'first_word'] = 'washington'

# Find matches and mismatches
DF1['match_found'] = DF1['first_word'].isin(DF2['first_word'])

# Results
mismatched = DF1[~DF1['match_found']]
matched = DF1[DF1['match_found']]

print(f"Total cities: {len(DF1)}")
print(f"Matched: {len(matched)}")
print(f"Mismatched: {len(mismatched)}")

print("\n=== MATCHED CITIES ===")
for city in matched['city_full']:
    print(f"✓ {city}")

print("\n=== MISMATCHED CITIES ===")
for city in mismatched['city_full']:
    print(f"✗ {city}")

# # Show what first words were compared
# print("\n=== COMPARISON DETAILS ===")
# for idx, row in mismatched.iterrows():
#     print(f"\n'{row['city_full']}' -> first word: '{row['first_word']}'")
#     print(f"Available first words in DF2: {sorted(DF2['first_word'].unique())}")

Total cities: 30
Matched: 27
Mismatched: 3

=== MATCHED CITIES ===
✓ Atlanta-Sandy Springs-Alpharetta
✓ Boston-Cambridge-Newton
✓ Tampa-St. Petersburg-Clearwater
✓ Baltimore-Columbia-Towson
✓ Portland-Vancouver-Hillsboro
✓ Philadelphia-Camden-Wilmington
✓ New York-Newark-Jersey City
✓ Chicago-Naperville-Elgin
✓ Orlando-Kissimmee-Sanford
✓ Seattle-Tacoma-Bellevue
✓ San Francisco-Oakland-Berkeley
✓ San Diego-Chula Vista-Carlsbad
✓ Austin-Round Rock-Georgetown
✓ Sacramento-Roseville-Folsom
✓ Phoenix-Mesa-Chandler
✓ Riverside-San Bernardino-Ontario
✓ San Antonio-New Braunfels
✓ Detroit-Warren-Dearborn
✓ Houston-The Woodlands-Sugar Land
✓ Charlotte-Concord-Gastonia
✓ Denver-Aurora-Lakewood
✓ Los Angeles-Long Beach-Anaheim
✓ DC_Metro
✓ Dallas-Fort Worth-Arlington
✓ Minneapolis-St. Paul-Bloomington
✓ Las Vegas-Henderson-Paradise
✓ Miami-Fort Lauderdale-Pompano Beach

=== MISMATCHED CITIES ===
✗ Pittsburgh
✗ St. Louis
✗ Cincinnati


In [44]:
# Take everything before first comma in DF2
DF2['city_clean'] = DF2['metro_full'].str.split(',').str[0].str.strip()

# Handle special cases
DF1['city_clean'] = DF1['city_full']
DF1.loc[DF1['city_full'] == 'DC_Metro', 'city_clean'] = 'Washington-Arlington-Alexandria'

# Find matches and mismatches
matched = []
mismatched = []

for city1 in DF1['city_clean']:
    if city1 in DF2['city_clean'].values:
        matched.append(city1)
    else:
        mismatched.append(city1)

# Results
print(f"Total cities: {len(DF1)}")
print(f"Matched: {len(matched)}")
print(f"Mismatched: {len(mismatched)}")

print("\n=== MATCHED CITIES ===")
for city in DF1[DF1['city_clean'].isin(matched)]['city_full']:
    print(f"✓ {city}")

print("\n=== MISMATCHED CITIES ===")
for city in DF1[DF1['city_clean'].isin(mismatched)]['city_full']:
    print(f"✗ {city}")

# Show detailed comparison for mismatched cities
print("\n=== DETAILED COMPARISON FOR MISMATCHES ===")
for city in mismatched:
    print(f"\nLooking for: '{city}'")
    
    # Show what DF2 has for similar cities
    first_word = city.split('-')[0].lower()
    similar = DF2[DF2['city_clean'].str.lower().str.contains(first_word, na=False)]
    
    if not similar.empty:
        print(f"Similar entries in DF2:")
        for _, row in similar.iterrows():
            print(f"  → {row['metro_full']}")
    else:
        print(f"No similar entries found in DF2")

Total cities: 30
Matched: 23
Mismatched: 7

=== MATCHED CITIES ===
✓ Pittsburgh
✓ Boston-Cambridge-Newton
✓ Tampa-St. Petersburg-Clearwater
✓ Baltimore-Columbia-Towson
✓ Portland-Vancouver-Hillsboro
✓ Philadelphia-Camden-Wilmington
✓ New York-Newark-Jersey City
✓ Chicago-Naperville-Elgin
✓ Orlando-Kissimmee-Sanford
✓ Seattle-Tacoma-Bellevue
✓ San Diego-Chula Vista-Carlsbad
✓ St. Louis
✓ Sacramento-Roseville-Folsom
✓ Phoenix-Mesa-Chandler
✓ Riverside-San Bernardino-Ontario
✓ San Antonio-New Braunfels
✓ Detroit-Warren-Dearborn
✓ Cincinnati
✓ Charlotte-Concord-Gastonia
✓ Los Angeles-Long Beach-Anaheim
✓ DC_Metro
✓ Dallas-Fort Worth-Arlington
✓ Minneapolis-St. Paul-Bloomington

=== MISMATCHED CITIES ===
✗ Atlanta-Sandy Springs-Alpharetta
✗ San Francisco-Oakland-Berkeley
✗ Austin-Round Rock-Georgetown
✗ Houston-The Woodlands-Sugar Land
✗ Denver-Aurora-Lakewood
✗ Las Vegas-Henderson-Paradise
✗ Miami-Fort Lauderdale-Pompano Beach

=== DETAILED COMPARISON FOR MISMATCHES ===

Looking for: 'Atla

# Mismatches and mappings:

=== DETAILED COMPARISON FOR MISMATCHES ===

Looking for: 'Atlanta-Sandy Springs-Alpharetta'
Similar entries in DF2:
  → Atlanta-Sandy Springs-Roswell, GA

Looking for: 'San Francisco-Oakland-Berkeley'
Similar entries in DF2:
  → San Francisco-Oakland-Fremont, CA

Looking for: 'Austin-Round Rock-Georgetown'
Similar entries in DF2:
  → Austin-Round Rock-San Marcos, TX

Looking for: 'Houston-The Woodlands-Sugar Land'
Similar entries in DF2:
  → Houston-Pasadena-The Woodlands, TX

Looking for: 'Denver-Aurora-Lakewood'
Similar entries in DF2:
  → Denver-Aurora-Centennial, CO

Looking for: 'Las Vegas-Henderson-Paradise'
Similar entries in DF2:
  → Las Vegas-Henderson-North Las Vegas, NV

Looking for: 'Miami-Fort Lauderdale-Pompano Beach'
Similar entries in DF2:
  → Miami-Fort Lauderdale-West Palm Beach, FL